# 01 — IR and PTX exploration (Phase 3)

Take two contrasting configs on one shape (fastest vs slowest valid, or spilling vs not), dump
TTIR → TTGIR → LLVM IR → PTX (→ SASS), and explain *with quoted snippets* why one is faster:
tensor-core `mma`, `ldmatrix`, `cp.async` (sm_80+ only), vector loads, unrolling, spills (`ld.local`/`st.local`).

Run after notebook 02 (uses the sweep to pick configs); falls back to a fixed pair otherwise.

In [ ]:
# --- ArchTune setup (same in every notebook) ---
import importlib.metadata as md, os, subprocess, sys
PROJECT = "/content/drive/MyDrive/archtune"
NEED_GPU_STACK = True

print("1/4 Google Drive")
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
assert os.path.isdir(PROJECT), f"project folder not found at {PROJECT} (see docs/COLAB_GUIDE.md step 1)"
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)   # import archtune straight from Drive, no pip install needed

def _ver(pkg):
    try:
        return md.version(pkg).split("+")[0]
    except md.PackageNotFoundError:
        return None

print("2/4 torch / triton:", _ver("torch"), _ver("triton"))
if NEED_GPU_STACK and (_ver("torch"), _ver("triton")) != ("2.6.0", "3.2.0"):
    print("    installing torch 2.6.0 + triton 3.2.0 (about 1 GB, a few minutes)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.6.0", "triton==3.2.0"], check=True)
    if "torch" in sys.modules:
        raise SystemExit("Installed. Now: Runtime > Restart session, then run this cell again.")

print("3/4 importing archtune")
from archtune import env
ENV = env.capture()
MINI = env.mini_mode()   # ARCHTUNE_MINI=1 -> tiny debug workload (set by run_all)
RESULTS = os.environ.get("ARCHTUNE_RESULTS") or os.path.join(PROJECT, "results_mini" if MINI else "results")

print("4/4 ready:", "MINI" if MINI else "FULL", "mode, results ->", RESULTS)
print(ENV)
if NEED_GPU_STACK and not env.pinned_ok(ENV):
    raise SystemExit("Wrong torch/triton loaded. Runtime > Restart session, then run this cell again.")

In [ ]:
from archtune import workloads, space
MODEL_CFG = workloads.load_model_config(cache_dir=RESULTS)
DIMS = workloads.derive_dims(MODEL_CFG)
MM_SHAPES = workloads.matmul_shapes(DIMS)
RMS_SHAPES = workloads.rmsnorm_shapes(DIMS)
SPLIT = {**workloads.train_test_split([s.shape_id for s in MM_SHAPES]),
         **workloads.train_test_split([s.shape_id for s in RMS_SHAPES])}
if MINI:
    MINI_SHAPES = workloads.mini_shapes(MM_SHAPES + RMS_SHAPES)
    MM_SHAPES = [s for s in MINI_SHAPES if s.kernel == "matmul"]
    RMS_SHAPES = [s for s in MINI_SHAPES if s.kernel == "rmsnorm"]
print(DIMS)
for s in MM_SHAPES + RMS_SHAPES:
    print(f"{s.shape_id:24s} {SPLIT[s.shape_id]:5s} {','.join(s.roles)}")

In [ ]:
import os, pandas as pd
from archtune import compile_info, explore, space, search
from archtune.kernels import matmul as mm
GPU_DIR = os.path.join(RESULTS, ENV["gpu_tag"])
SHAPE = next(s for s in MM_SHAPES if s.shape_id == "mm_M4096_N4096_K4096")
try:
    sw = search.prepare(pd.read_parquet(os.path.join(GPU_DIR, "sweep.parquet")))
    t = sw[(sw.shape_id == SHAPE.shape_id) & sw.valid].sort_values("time_ms_median")
    knob = [c for c in t.columns if c.startswith("cfg_") and c != "cfg_hash"]
    pick = lambda r: {c[4:]: int(r[c]) for c in knob if pd.notna(r[c])}
    FAST, SLOW = pick(t.iloc[0]), pick(t.iloc[-1])
    print("fast", t.iloc[0].time_ms_median, "ms", FAST); print("slow", t.iloc[-1].time_ms_median, "ms", SLOW)
except FileNotFoundError:
    FAST = {"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 128, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8, "num_warps": 4, "num_stages": 3}
    SLOW = {"BLOCK_SIZE_M": 256, "BLOCK_SIZE_N": 256, "BLOCK_SIZE_K": 64, "GROUP_SIZE_M": 8, "num_warps": 2, "num_stages": 2}

In [ ]:
compile_info.fresh_cache_dir()
a, b, c = mm.make_inputs(SHAPE.M, SHAPE.N, SHAPE.K)
LISTINGS, RES = {}, {}
for name, cfg in (("fast", FAST), ("slow", SLOW)):
    meta, launch = space.split_launch(cfg)
    r = compile_info.compile_config(mm.matmul_kernel, mm.kernel_args(a, b, c), mm.grid(SHAPE.M, SHAPE.N, cfg), meta, launch)
    sass = compile_info.disassemble(r.asm.get("cubin")) if r.asm.get("cubin") else None
    explore.dump_ir(r.asm, os.path.join(GPU_DIR, "ir_exploration"), name, sass)
    LISTINGS[name] = {"ptx": r.asm.get("ptx"), "sass": sass}; RES[name] = r
    print(name, r.status, "regs", r.n_regs, "spills", r.n_spills, "smem", r.shared_bytes)
explore.compare_features(LISTINGS)

## TTGIR layouts (how Triton maps the tile onto threads and tensor cores)

In [ ]:
for name in ("fast", "slow"):
    print(f"--- {name}"); print("\n".join(explore.ttgir_layouts(RES[name].asm.get("ttgir", ""))))

## PTX snippets: tensor cores, ldmatrix, async copies, vector loads, spills

In [ ]:
for name in ("fast", "slow"):
    ptx = LISTINGS[name]["ptx"]
    for pat in (r"mma\.sync", r"ldmatrix", r"cp\.async\.c[ag]", r"ld\.global.*\.v[248]", r"(ld|st)\.local"):
        print(f"=== {name}: {pat}\n{explore.snippets(ptx, pat, context=0, max_hits=2)}\n")

## Explanation (write after running — quote the snippets above)

*Template — replace with observations from your run; do not keep claims the data does not show.*

1. **Tensor-core instruction mix:** compare `ptx_mma` counts and the `mma.sync` shape (e.g. `m16n8k8` on sm_75 vs `m16n8k16` on sm_80+).
2. **Operand staging:** `ldmatrix` present? On sm_80+ `cp.async` copies global→shared without registers; on sm_75 there is no `cp.async`, so pipelining goes through registers.
3. **Spills:** any `ld.local`/`st.local` in the slow config? Tie to `n_spills` and the register count.
4. **Occupancy / waves:** use `predict.occupancy(n_regs, shared_bytes, num_warps, arch)` and the tile count to explain the remaining gap.